# CPHD Metadata Inspection

**Source script:** `grdl/example/image_processing/sar/dump_cphd_metadata.py`

This notebook demonstrates how to open a CPHD (Compensated Phase History Data) file
and extract all metadata fields for inspection. CPHD is the NGA standard for distributing
compensated radar phase history — the raw material for SAR image formation.

## What you will learn

- How to open a CPHD file with the IO factory pattern: `get_reader('cphd', path)`
- The structure of CPHD metadata: collection info, global parameters, channels, waveform, PVP
- How to derive physical quantities from PVP arrays (PRF, platform velocity, slant range)
- How to detect collection mode (spotlight vs. stripmap) from SRP drift

## Data Setup

Place your CPHD file path in the `CPHD_PATH` variable in Cell 3 below.
This notebook was validated against Umbra and Capella CPHD products.

**Default test path:** `/data/sar/from_duane/CPHD/` — adjust for your environment.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np
from numpy.linalg import norm

from grdl.IO import get_reader

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Set your CPHD file path here
# ════════════════════════════════════════════════════════════════════
CPHD_PATH = Path("/data/sar/from_duane/CPHD/2023-08-25-02-36-21_UMBRA-05_CPHD.cphd")

assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
print(f"Target: {CPHD_PATH.name} ({CPHD_PATH.stat().st_size / 1e9:.2f} GB)")

## 1. Collection Info

The `collection_info` block identifies the sensor, collection mode, and classification.
This is the first thing you check to determine which IFP (Image Formation Processor)
to use downstream:

| `radar_mode` | Algorithm |
|---|---|
| SPOTLIGHT | PFA (Polar Format Algorithm) |
| STRIPMAP | RDA (Range-Doppler Algorithm) |
| DYNAMIC STRIPMAP | RDA or StripmapPFA |

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    backend = reader.backend

print(f"Backend: {backend}")
print()

ci = meta.collection_info
if ci is not None:
    print(f"  Collector Name : {ci.collector_name}")
    print(f"  Core Name      : {ci.core_name}")
    print(f"  Classification : {ci.classification}")
    print(f"  Collect Type   : {ci.collect_type}")
    print(f"  Radar Mode     : {ci.radar_mode}")
    print(f"  Radar Mode ID  : {ci.radar_mode_id}")
else:
    print("  (no collection info)")

## 2. Global Parameters

Defines the signal domain (FX = frequency domain), center frequency, bandwidth,
and the critical `phase_sgn` (±1) that controls IFP kernel conjugation.

In [ ]:
gp = meta.global_params
if gp is not None:
    print(f"  Domain Type        : {gp.domain_type}")
    print(f"  Phase SGN          : {gp.phase_sgn:+d}")
    print(f"  FX Band Min (Hz)   : {gp.fx_band_min}")
    print(f"  FX Band Max (Hz)   : {gp.fx_band_max}")
    print(f"  Bandwidth (Hz)     : {gp.bandwidth}")
    print(f"  Center Freq (Hz)   : {gp.center_frequency}")
    if gp.center_frequency:
        print(f"  Center Freq (GHz)  : {gp.center_frequency / 1e9:.6f}")
        wavelength = 299792458.0 / gp.center_frequency
        print(f"  Wavelength (m)     : {wavelength:.6f}")
    print(f"  TOA Swath Min (s)  : {gp.toa_swath_min}")
    print(f"  TOA Swath Max (s)  : {gp.toa_swath_max}")
else:
    print("  (no global params)")

## 3. Channel & Waveform Parameters

Each CPHD channel contains:
- `num_vectors` — number of transmitted/received pulses
- `num_samples` — number of range samples per pulse

The signal array shape is `(num_vectors, num_samples)` complex64.

The transmit waveform defines the LFM (Linear Frequency Modulated) chirp:
- `lfm_rate × pulse_length = chirp_bandwidth`

In [ ]:
# Channels
print(f"  Num Channels: {meta.num_channels}")
for i, ch in enumerate(meta.channels):
    print(f"  --- Channel {i} ---")
    print(f"    Identifier      : {ch.identifier}")
    print(f"    Num Vectors     : {ch.num_vectors}")
    print(f"    Num Samples     : {ch.num_samples}")
    print(f"    Signal shape    : ({ch.num_vectors}, {ch.num_samples}) complex64")
    print(f"    Signal size     : {ch.num_vectors * ch.num_samples * 8 / 1e9:.2f} GB")

print()

# Transmit Waveform
tx = meta.tx_waveform
if tx is not None:
    print(f"  TX Identifier     : {tx.identifier}")
    print(f"  LFM Rate (Hz/s)   : {tx.lfm_rate}")
    print(f"  Pulse Length (s)   : {tx.pulse_length}")
    if tx.lfm_rate and tx.pulse_length:
        chirp_bw = tx.lfm_rate * tx.pulse_length
        print(f"  Chirp BW (MHz)    : {chirp_bw / 1e6:.4f}")

# Receive Parameters
rcv = meta.rcv_parameters
if rcv is not None:
    print(f"  RCV Window (s)    : {rcv.window_length}")
    print(f"  Sample Rate (MHz) : {rcv.sample_rate / 1e6:.4f}")

## 4. Per-Vector Parameters (PVP)

PVP arrays store per-pulse metadata: timing, platform position/velocity,
frequency support, and scene reference point (SRP).

Key derived quantities:
- **PRF** = 1 / mean(Δt) where Δt is the inter-pulse interval
- **SRP drift** = 0 for spotlight (fixed scene center), >0 for stripmap
- **Slant range** = |ARP - SRP| = distance from platform to scene center

In [ ]:
pvp = meta.pvp
assert pvp is not None, "No PVP data in this CPHD file"

print(f"  Num Vectors       : {pvp.num_vectors}")
print(f"  First Valid Pulse : {pvp.first_valid_pulse}")

# Timing
if pvp.tx_time is not None:
    print(f"\n  --- Timing ---")
    print(f"  TX Time Span (s)  : {pvp.tx_time.max() - pvp.tx_time.min():.9f}")
    mid_times = 0.5 * (pvp.tx_time + pvp.rcv_time)
    dt = np.diff(mid_times)
    prf = 1.0 / np.mean(dt)
    print(f"  Avg Pulse Interval: {np.mean(dt):.9f} s")
    print(f"  Avg PRF           : {prf:.2f} Hz")
    print(f"  Collection Time   : {pvp.tx_time.max() - pvp.tx_time.min():.3f} s")

In [ ]:
# SRP Position — determines spotlight vs stripmap
if pvp.srp_pos is not None:
    print("  --- SRP Position (ECF, meters) ---")
    print(f"  SRP[0]            : ({pvp.srp_pos[0, 0]:.3f}, {pvp.srp_pos[0, 1]:.3f}, {pvp.srp_pos[0, 2]:.3f})")
    print(f"  SRP[-1]           : ({pvp.srp_pos[-1, 0]:.3f}, {pvp.srp_pos[-1, 1]:.3f}, {pvp.srp_pos[-1, 2]:.3f})")
    drift = norm(pvp.srp_pos[-1] - pvp.srp_pos[0])
    print(f"  SRP Total Drift   : {drift:.3f} m")
    per_pulse_drift = norm(np.diff(pvp.srp_pos, axis=0), axis=1)
    print(f"  Drift/Pulse [mean]: {per_pulse_drift.mean():.6f} m")
    print()
    if drift > 1.0:
        print("  ⚡ MODE DETECTION: STRIPMAP (SRP drifts significantly)")
    else:
        print("  ⚡ MODE DETECTION: SPOTLIGHT (SRP approximately fixed)")

In [ ]:
# Platform state vectors
if pvp.tx_pos is not None:
    arp = pvp.tx_pos  # monostatic: tx_pos == ARP
    print("  --- Platform Position (ECF, meters) ---")
    print(f"  ARP[0]            : ({arp[0, 0]:.3f}, {arp[0, 1]:.3f}, {arp[0, 2]:.3f})")
    print(f"  ARP[-1]           : ({arp[-1, 0]:.3f}, {arp[-1, 1]:.3f}, {arp[-1, 2]:.3f})")
    arp_mag = norm(arp, axis=1)
    print(f"  |R| range         : [{arp_mag.min():.1f}, {arp_mag.max():.1f}] m")
    print(f"  Approx altitude   : {(arp_mag.mean() - 6371000):.1f} m")
    print(f"  Track length      : {norm(arp[-1] - arp[0]):.3f} m")

if pvp.tx_vel is not None:
    vel = pvp.tx_vel
    vel_mag = norm(vel, axis=1)
    print(f"\n  --- Platform Velocity (ECF, m/s) ---")
    print(f"  Speed [min]       : {vel_mag.min():.3f} m/s")
    print(f"  Speed [max]       : {vel_mag.max():.3f} m/s")
    print(f"  Speed [mean]      : {vel_mag.mean():.3f} m/s")

In [ ]:
# Frequency support (per-pulse)
print("  --- Frequency Parameters ---")
if pvp.fx1 is not None and pvp.fx2 is not None:
    bw = pvp.fx2 - pvp.fx1
    print(f"  FX1 range (Hz)    : [{pvp.fx1.min():.2f}, {pvp.fx1.max():.2f}]")
    print(f"  FX2 range (Hz)    : [{pvp.fx2.min():.2f}, {pvp.fx2.max():.2f}]")
    print(f"  Per-Pulse BW (MHz): [{bw.min()/1e6:.4f}, {bw.max()/1e6:.4f}]")
    print(f"  Mean BW (MHz)     : {bw.mean()/1e6:.4f}")

# Signal validity
if pvp.signal is not None:
    n_valid = int(np.sum(pvp.signal > 0))
    print(f"\n  --- Signal Validity ---")
    print(f"  Valid Pulses      : {n_valid} / {pvp.num_vectors}")
    print(f"  Signal flags      : [{pvp.signal.min()}, {pvp.signal.max()}]")

## 5. Derived Geometry

Compute slant range from platform (ARP) to scene center (SRP).
This tells you the imaging geometry: closer range = higher resolution
in range dimension for a given bandwidth.

In [ ]:
if pvp.srp_pos is not None and pvp.tx_pos is not None:
    r_srp_arp = norm(pvp.tx_pos - pvp.srp_pos, axis=1)
    print(f"  Slant Range [min]  : {r_srp_arp.min():.3f} m")
    print(f"  Slant Range [max]  : {r_srp_arp.max():.3f} m")
    print(f"  Slant Range [mean] : {r_srp_arp.mean():.3f} m ({r_srp_arp.mean()/1000:.3f} km)")
    print()
    # Theoretical range resolution: c / (2 * BW)
    if gp is not None and gp.bandwidth:
        c = 299792458.0
        range_res = c / (2.0 * gp.bandwidth)
        print(f"  Theoretical range resolution: {range_res:.3f} m")
        print(f"  (from bandwidth {gp.bandwidth/1e6:.2f} MHz)")

## 6. PVP Array Samples

Inspect the first and last few PVP entries to verify data continuity
and check for anomalies (time jumps, position discontinuities, etc.).

In [ ]:
n = pvp.num_vectors
sample_indices = list(range(min(5, n))) + list(range(max(0, n - 3), n))
sample_indices = sorted(set(sample_indices))

print(f"PVP samples (indices {sample_indices}):")
print(f"{'idx':>6}  {'tx_time':>14}  {'speed_m/s':>10}  {'slant_range_m':>14}")
print("-" * 52)

for idx in sample_indices:
    t = pvp.tx_time[idx] if pvp.tx_time is not None else float('nan')
    spd = norm(pvp.tx_vel[idx]) if pvp.tx_vel is not None else float('nan')
    sr = norm(pvp.tx_pos[idx] - pvp.srp_pos[idx]) if (pvp.tx_pos is not None and pvp.srp_pos is not None) else float('nan')
    print(f"{idx:6d}  {t:14.6f}  {spd:10.3f}  {sr:14.3f}")

---

## Summary

This notebook replaces `grdl/example/image_processing/sar/dump_cphd_metadata.py`.

Key patterns demonstrated:
- **IO factory pattern**: `get_reader('cphd', path)` replaces direct `CPHDReader` import (recommended approach)
- **Context manager**: `with get_reader(...) as reader:` ensures file handles are released
- **Metadata access**: All fields available via `reader.metadata` without reading signal data
- **PVP derivation**: PRF, slant range, collection mode all derived from PVP arrays
- **No sys.path hacks**: Clean import via editable install